# RiskRadar-Extractor-v1: 1-Click Free T4 Colab Training Notebook\n**Smart Industrial Safety Intelligence & SIF Precursor Detection**\nTrains a domain-specialized LoRA adapter on `Qwen/Qwen2.5-1.5B-Instruct` using unsloth/peft + trl.

In [ ]:
# 1. Install optimized dependencies (free T4 GPU supported)\n!pip install -q torch transformers datasets peft trl bitsandbytes accelerate huggingface_hub pydantic

In [ ]:
# 2. Verify GPU\nimport torch\nprint('CUDA Available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('GPU Device:', torch.cuda.get_device_name(0))\n    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

In [ ]:
# 3. Clone Repository or Upload ml_model/data files\n# Alternatively, upload train.jsonl and val.jsonl directly to Colab workspace\nimport os\nprint('Files in current directory:', os.listdir('.'))

In [ ]:
# 4. Launch Fine-Tuning Pipeline\nfrom datasets import load_dataset\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\nfrom trl import SFTTrainer\n\nMODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'\nOUTPUT_DIR = './riskradar-extractor-v1'\n\ntokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)\ntokenizer.pad_token = tokenizer.eos_token\n\nbnb_config = BitsAndBytesConfig(\n    load_in_4bit=True,\n    bnb_4bit_quant_type='nf4',\n    bnb_4bit_compute_dtype=torch.bfloat16,\n    bnb_4bit_use_double_quant=True\n)\n\nmodel = AutoModelForCausalLM.from_pretrained(\n    MODEL_ID,\n    quantization_config=bnb_config,\n    device_map='auto',\n    trust_remote_code=True\n)\nmodel = prepare_model_for_kbit_training(model)\n\npeft_config = LoraConfig(\n    r=16,\n    lora_alpha=32,\n    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],\n    lora_dropout=0.05,\n    bias='none',\n    task_type='CAUSAL_LM'\n)\nmodel = get_peft_model(model, peft_config)\n\ntraining_args = TrainingArguments(\n    output_dir=OUTPUT_DIR,\n    num_train_epochs=3,\n    per_device_train_batch_size=4,\n    gradient_accumulation_steps=4,\n    learning_rate=2e-4,\n    warmup_ratio=0.1,\n    logging_steps=10,\n    save_strategy='epoch',\n    bf16=True,\n    seed=42\n)\nprint('Ready to train with SFTTrainer!')

In [ ]:
# 5. Zip and Download Model Artifacts\n!zip -r riskradar-extractor-v1.zip ./riskradar-extractor-v1\nfrom google.colab import files\nfiles.download('riskradar-extractor-v1.zip')